In [1]:
import pandas as pd
import holidays

def load_daily_exits(path):
    df = pd.read_csv(
        path,
        names=["date", "hour", "origin", "dest", "trips"],
        usecols=["date", "dest", "trips"],          # skip columns we don't need
        dtype={"dest": "category", "trips": "int32"},
    )
    return df.groupby(["date", "dest"], observed=True)["trips"].sum().reset_index()

daily = pd.concat([
    load_daily_exits("../data/raw/date-hour-soo-dest-2024.csv.gz"),
    load_daily_exits("../data/raw/date-hour-soo-dest-2025.csv.gz"),
])
daily = daily.rename(columns={"dest": "station", "trips": "exits"})
daily["date"] = pd.to_datetime(daily["date"])
daily.head()

,date,station,exits
0,2024-01-01,12TH,1181
1,2024-01-01,16TH,1994
2,2024-01-01,19TH,1291
3,2024-01-01,24TH,1877
4,2024-01-01,ANTC,347


In [2]:
top = daily.groupby("station", observed=True)["exits"].sum().nlargest(8).index
daily = daily[daily["station"].isin(top)].copy()
daily["station"] = daily["station"].astype(str)
print(list(top))

['EMBR', 'MONT', 'POWL', 'CIVC', 'DBRK', '16TH', '24TH', '12TH']


In [3]:
weather = pd.read_csv("../data/raw/weather.csv", parse_dates=["date"])
df = daily.merge(weather, on="date", how="left")
df.isna().sum()   # should be all zeros

date          0
station       0
exits         0
temp_max_f    0
precip_in     0
dtype: int64

In [4]:
us_holidays = holidays.US(years=[2024, 2025])

df["day_of_week"] = df["date"].dt.day_name()
df["month"] = df["date"].dt.month
df["is_holiday"] = df["date"].dt.date.isin(us_holidays).astype(int)

df.sample(10)

,date,station,exits,temp_max_f,precip_in,day_of_week,month,is_holiday
1584,2024-07-17,12TH,5623,77.4,0.000,Wednesday,7,0
1346,2024-06-17,24TH,4710,84.2,0.000,Monday,6,0
5770,2025-12-22,24TH,3472,61.8,1.146,Monday,12,0
5778,2025-12-23,24TH,3545,61.8,0.374,Tuesday,12,0
5234,2025-10-16,24TH,6012,66.6,0.000,Thursday,10,0
2434,2024-10-31,24TH,6366,64.1,0.091,Thursday,10,0
2962,2025-01-05,24TH,2382,60.2,0.000,Sunday,1,0
4063,2025-05-22,POWL,12419,73.8,0.000,Thursday,5,0
3148,2025-01-28,DBRK,5845,58.5,0.000,Tuesday,1,0
189,2024-01-24,EMBR,17842,59.0,0.185,Wednesday,1,0


In [5]:
df.to_csv("../data/processed/bart_daily.csv", index=False)
print(df.shape)

(5848, 8)


In [6]:
import sys
sys.path.append("../src")

import pandas as pd
from linreg.preprocessing import split_by_date, fit_one_hot, transform_one_hot

df = pd.read_csv("../data/processed/bart_daily.csv")
train, test = split_by_date(df)

print(train['date'].max(), test['date'].min())
print(len(train) + len(test) == len(df))


2024-12-31 00:00:00 2025-01-01 00:00:00
True


In [7]:
categories = fit_one_hot(train)
train_enc = transform_one_hot(train, categories)
test_enc = transform_one_hot(test, categories)

print(list(train_enc.columns.tolist()))
print(train[["day_of_week", "month", "station"]].nunique())
print(list(train_enc.columns) == list(test_enc.columns))
print(train_enc.shape[1])



['date', 'exits', 'temp_max_f', 'precip_in', 'is_holiday', 'day_of_week_Monday', 'day_of_week_Saturday', 'day_of_week_Sunday', 'day_of_week_Thursday', 'day_of_week_Tuesday', 'day_of_week_Wednesday', 'month_2', 'month_3', 'month_4', 'month_5', 'month_6', 'month_7', 'month_8', 'month_9', 'month_10', 'month_11', 'month_12', 'station_16TH', 'station_24TH', 'station_CIVC', 'station_DBRK', 'station_EMBR', 'station_MONT', 'station_POWL']
day_of_week     7
month          12
station         8
dtype: int64
True
29
